# Reproducible Execution Notebook: Frequent Itemset Mining Using Eclat (Julia)

This notebook provides an end-to-end reproducible pipeline for verifying correctness, benchmarking performance, and reproducing the real-world applications of the `EclatFIM` Julia package on Google Colab.

- **Course:** CSC14004 -- Data Mining and Applications (Class `24KHMT1`, Semester 1, Academic Year 2026--2027), Faculty of Information Technology, University of Science, VNU-HCM.
- **Algorithm:** Eclat (M. J. Zaki, *IEEE Transactions on Knowledge and Data Engineering*, vol. 12, no. 3, pp. 372--390, 2000).
- **Authors:** Phạm Trần Anh Quân (`24127226` -- Core Algorithm, Optimization, Application 2), Nguyễn Tiến Cường (`24127337` -- SPMF I/O, Verification, Benchmarking Infrastructure, Application 1).
- **Execution Mode:** `MODE = "quick"` executes a representative subset of the parameter grid within the free Colab session budget; `MODE = "full"` executes the complete experimental grid.

Each section is self-contained to ensure modular verification and reproducibility.

In [ ]:
# Reproducibility configuration
import os

MODE = "quick"  # Valid options: "quick" or "full"
REPO_URL = "https://github.com/24127226/16-FIM-DATA-MINING.git"
REPO_BRANCH = "main"  # Target branch or release tag (e.g., "v1.0")
REPO_DIR = "/content/16-FIM-DATA-MINING"
JULIA_VERSION = "1.11.4"
os.environ["FIM_DATA_BASE"] = "https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main"

print(f"Mode: {MODE} | Repository: {REPO_URL} ({REPO_BRANCH})")

## Section 1. Environment Provisioning (Julia 1.11.4, Java, and SPMF)

Clones the repository, installs the official Julia `1.11.4` runtime, instantiates locked dependencies from `Manifest.toml`, verifies the Java runtime, downloads `spmf.jar` with SHA-256 verification, and records hardware specifications.

In [ ]:
import os, shutil, subprocess

# Clone repository if not already present
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Install Julia 1.11.4
julia_bin = f"/usr/local/julia-{JULIA_VERSION}/bin/julia"
if not os.path.exists(julia_bin) and shutil.which("julia") is None:
    url = f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
    subprocess.run(f"curl -fsSL {url} | tar -xz -C /usr/local", shell=True, check=True)
    subprocess.run(["ln", "-sf", julia_bin, "/usr/local/bin/julia"], check=True)

!julia --version
!julia --project=. -e "using Pkg; Pkg.instantiate(); Pkg.precompile()"

In [ ]:
# Verify Java runtime, download SPMF reference binary, and record machine metadata
!java -version || true
!julia --project=. scripts/fetch_spmf.jl
!julia --project=. scripts/env_info.jl

## Section 2. Benchmark Dataset Acquisition and SHA-256 Verification

Downloads standard FIM benchmark datasets (`chess`, `mushroom`, `retail`, `accidents`, `T10I4D100K`) from the public Hugging Face repository (no authentication required) and verifies SHA-256 checksums.

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl benchmark

## Section 3. Command-Line Interface Verification on Canonical Toy Datasets

Executes `demo.jl` on `pasquier99.txt` at `minsup = 40%` (expected: 15 frequent itemsets) and `bookstore.txt` at `minsup = 0.5` (expected: 19 frequent itemsets with internal `EclatStats` counters).

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. demo.jl --minsup 40% -o - data/toy/pasquier99.txt
!julia --project=. demo.jl --minsup 0.5 --stats -o output_bookstore.txt data/toy/bookstore.txt

## Section 4. Three-Tier Automated Test Suite (`test/runtests.jl`)

Executes the complete automated verification suite: SPMF I/O and `minsup_count` boundary tests, 200-database brute-force oracle verification, canonical datasets, edge cases, association rule tests (`rules.jl`), and cross-variant benchmark checks.

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. test/runtests.jl

## Section 5. Chapter 2 -- Experimental Performance Evaluation

Executes `scripts/bench.jl` under the isolated-process measurement protocol with warm-up and exports results to the standardized 17-column CSV schema.

In [ ]:
import os
os.chdir(REPO_DIR)
mode_arg = "quick" if MODE == "quick" else "full"
repeats = 1 if MODE == "quick" else 3
!julia --project=. scripts/bench.jl --mode {mode_arg} --variants v0,v1 --repeats {repeats} --out results/bench_colab.csv
!head -n 20 results/bench_colab.csv

## Section 6. Chapter 3 -- Application 1: Market Basket Analysis on Groceries

Reproduces the full Application 1 pipeline on `groceries.csv` (9,835 transactions, 169 items): quantitative comparison of two data transformation schemes (Table B08), threshold sensitivity grid, top-10 association rules ranked by lift (Table B09), and out-of-sample 80/20 stability evaluation (Table B10).

In [ ]:
import os
from IPython.display import Markdown, display

os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl groceries
!julia --project=. apps/groceries/run_groceries.jl

with open("results/groceries_summary.md", "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

## Section 7. Chapter 3 -- Application 2: Visual Pattern Discovery via Bag of Visual Words

Execution cells for Application 2 on the Caltech-101 image dataset will be integrated upon completion of `apps/bovw/`.

## Section 8. Report and Artifacts

The complete project report is stored at `docs/Report.pdf` in this repository.